In [1]:
import time
import random
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
def get_movie_urls():
    """
    获取豆瓣 Top 250 电影的 URL 列表，每页 25 部电影，共 10 页。
    在循环中使用进度条显示获取进度。
    """
    movie_urls = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/97.0.4692.99 Safari/537.36',
        'Accept-Language': 'zh-CN,zh;q=0.9'
    }
    # 使用 tqdm 显示页码进度
    for start in tqdm(range(0, 250, 25), desc="获取电影列表"):
        url = f'https://movie.douban.com/top250?start={start}'
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        for tag in soup.find_all('div', class_='hd'):
            movie_urls.append(tag.a['href'])
        # 随机延时 1~2 秒，降低反爬风险
        time.sleep(random.uniform(1, 2))
    return movie_urls

In [3]:
movie_urls=get_movie_urls()

获取电影列表:   0%|          | 0/10 [00:00<?, ?it/s]

In [4]:
movie_urls

['https://movie.douban.com/subject/1292052/',
 'https://movie.douban.com/subject/1291546/',
 'https://movie.douban.com/subject/1292722/',
 'https://movie.douban.com/subject/1292720/',
 'https://movie.douban.com/subject/1291561/',
 'https://movie.douban.com/subject/1292063/',
 'https://movie.douban.com/subject/1889243/',
 'https://movie.douban.com/subject/1295644/',
 'https://movie.douban.com/subject/3541415/',
 'https://movie.douban.com/subject/1292064/',
 'https://movie.douban.com/subject/1295124/',
 'https://movie.douban.com/subject/3011091/',
 'https://movie.douban.com/subject/1292001/',
 'https://movie.douban.com/subject/25662329/',
 'https://movie.douban.com/subject/3793023/',
 'https://movie.douban.com/subject/2131459/',
 'https://movie.douban.com/subject/1291549/',
 'https://movie.douban.com/subject/1307914/',
 'https://movie.douban.com/subject/1296141/',
 'https://movie.douban.com/subject/20495023/',
 'https://movie.douban.com/subject/1292213/',
 'https://movie.douban.com/subje

In [5]:
def get_reviews(movie_url):
    """
    获取一部电影的三种类型短评：
      - 普通评论：f'{movie_url}comments?'
      - 中评：f'{movie_url}comments?percent_type=m'
      - 差评：f'{movie_url}comments?percent_type=l'
    每个页面大约返回 20 条短评。
    """
    reviews = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/97.0.4692.99 Safari/537.36',
        'Accept-Language': 'zh-CN,zh;q=0.9',
        'Referer': movie_url,  # 设置 Referer 模拟正常访问
        'Cookie':'bid=9itG4BMMPxA; _pk_id.100001.4cf6=083a8ca71192389a.1773822254.; ap_v=0,6.0; __yadk_uid=3GegGFKHnvUF1KBOvQFpI91wuq2sGqDw; ll="118254"; _vwo_uuid_v2=D5646A8F66ACACC33AE46A890B38B10E6|87dd6f47855688da414760b8a7c242f9; _pk_ref.100001.4cf6=%5B%22%22%2C%22%22%2C1773828950%2C%22https%3A%2F%2Fsec.douban.com%2F%22%5D; _pk_ses.100001.4cf6=1; __utma=30149280.1931089290.1773828950.1773828950.1773828950.1; __utmc=30149280; __utmz=30149280.1773828950.1.1.utmcsr=sec.douban.com|utmccn=(referral)|utmcmd=referral|utmcct=/; __utma=223695111.1186823090.1773828950.1773828950.1773828950.1; __utmb=223695111.0.10.1773828950; __utmc=223695111; __utmz=223695111.1773828950.1.1.utmcsr=sec.douban.com|utmccn=(referral)|utmcmd=referral|utmcct=/; dbcl2="294229929:PohCgPw+hy4"; ck=GQVC; push_noty_num=0; push_doumail_num=0; __utmt=1; __utmv=30149280.29422; __utmb=30149280.2.10.1773828950; frodotk_db="053888f5bdf4056d0fa99ae139dc1047"; RT=nu=https%3A%2F%2Fwww.douban.com%2F&cl=1773829153100&r=https%3A%2F%2Fwww.douban.com%2F&ul=1773829166029&hd=1773829166031'
    }
    # 三种页面参数，空字符串代表普通评论
    endpoints = ['', 'percent_type=m', 'percent_type=l']
    
    for ep in endpoints:
        # 构建完整的评论页面 URL
        url = f"{movie_url}comments?"
        if ep:
            url += ep
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        comment_list = soup.find_all('div', class_='comment')
        for comment in comment_list:
            
            # 获取评论内容
            review = comment.p.text.strip()
            
            # 获取评分信息，若没有评分则记为“无评分”
            rating_tag = comment.find('span', class_='rating')
            if rating_tag and 'class' in rating_tag.attrs:
            # rating_tag 的 class 例如 'allstar50'，其中 50 表示 5 分
                rating_class = rating_tag['class'][0]
                try:
                    rating = int(rating_class.replace('allstar', '')) // 10
                except Exception:
                    rating = None
            else:
                rating = None
            reviews.append((review, rating))
        # 每个页面请求后随机延时 1~3 秒
        time.sleep(random.uniform(1, 3))
    return reviews

In [6]:
data=[]
reviews = get_reviews(movie_urls[0])

In [7]:
reviews

[('恐惧让你沦为囚犯，希望让你重获自由。——《肖申克的救赎》', 5),
 ('人的生命不过是从一个洞穴通往另一个世界..然后在那个世界的雨中继续颤抖.i hope', 4),
 ('当年的奥斯卡颁奖礼上，被如日中天的《阿甘正传》掩盖了它的光彩，而随着时间的推移，这部电影在越来越多的人们心中的地位已超越了《阿甘》。每当现实令我疲惫得产生无力感，翻出这张碟，就重获力量。毫无疑问，本片位列男人必看的电影前三名！回顾那一段经典台词：“有的人的羽翼是如此光辉，即使世界上最黑暗的牢狱，也无法长久地将他围困！”',
  5),
 ('策划了19年的私奔……', 5),
 ('关于希望最强有力的注释。', 5),
 ('“这是一部男人必看的电影。”人人都这么说。但单纯从性别区分，就会让这电影变狭隘。《肖申克的救赎》突破了男人电影的局限，通篇几乎充满令人难以置信的温馨基调，而电影里最伟大的主题是“希望”。\r\n当我们无奈地遇到了如同肖申克一般囚禁了心灵自由的那种囹圄，我们是无奈的老布鲁克，灰心的瑞德，还是智慧的安迪？运用智慧，信任希望，并且勇敢面对恐惧心理，去打败它？\r\n经典的电影之所以经典，因为他们都在做同一件事——让你从不同的角度来欣赏希望的美好。',
  5),
 ('超级喜欢超级喜欢,不看的话人生不圆满.', 5),
 ('这样的男人谁会舍得背叛。。。', 5),
 ('一部没有爱情与美女的电影,却光芒四射', 5),
 ('Fear Can Hold You Prisoner, Hope Can Set You Free', 5),
 ('没有人会不喜欢吧！书和电影都好。', 5),
 ('Hope is a good thing, and maybe the best thing of all.', 4),
 ('还记得老brooks上吊自杀时心里有多难过。自由来的太晚，生命早已自行放弃。', 5),
 ('绝对是期待过高导致有点失望', 4),
 ('在我的心目中,它一直都是最被高估的电影。', 3),
 ('不愧是好莱坞大片，一环扣一环，没想到最后不但逃生生天还偷偷把典狱长的贪污转移了出来，证据也送达了出去，帮助了曾经关心过自己的人，情节安排的巧妙，演员演绎的也好，是美国的，有爱国情节的我给打四分',
  4),
 ('当雷电劈开圣经扉页的刹那，污水管道的血

In [8]:
for review, rating in reviews:
     data.append({

          'Review': review,
          'Rating': rating
     })

In [9]:
df=pd.DataFrame(data)

In [10]:
df

,Review,Rating
0,恐惧让你沦为囚犯，希望让你重获自由。——《肖申克的救赎》,5
1,人的生命不过是从一个洞穴通往另一个世界..然后在那个世界的雨中继续颤抖.i hope,4
2,当年的奥斯卡颁奖礼上，被如日中天的《阿甘正传》掩盖了它的光彩，而随着时间的推移，这部电影在越...,5
3,策划了19年的私奔……,5
4,关于希望最强有力的注释。,5
5,“这是一部男人必看的电影。”人人都这么说。但单纯从性别区分，就会让这电影变狭隘。《肖申克的救...,5
6,"超级喜欢超级喜欢,不看的话人生不圆满.",5
7,这样的男人谁会舍得背叛。。。,5
8,"一部没有爱情与美女的电影,却光芒四射",5
9,"Fear Can Hold You Prisoner, Hope Can Set You Free",5
